# 01 — Data exploration

What the data actually is, before any modelling: where it comes from, what is wrong with it, and what cleaning did about that.

> **Provenance.** Rates are *indicative mid-market* quotes from a public market-data provider — **not** the official rate published by the Banque Centrale de Mauritanie. EUR/MRU is a **derived cross** (USD/MRU × EUR/USD), not an observed quote.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.io as pio

pio.renderers.default = "notebook_connected"  # CDN reference, not an embedded copy of plotly.js per figure
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from mru_forecast import viz
from mru_forecast.config import load_config
from mru_forecast.data.cleaning import load_processed

config = load_config()
frame = load_processed(config)
TARGETS = [t["name"] for t in config["targets"]]
print(f"{len(frame)} business days: {frame.index.min().date()} to {frame.index.max().date()}")
frame.tail(3).round(4)


2053 business days: 2018-11-01 to 2026-09-14


,BRENT_OIL,EUR_MRU,EUR_USD,GOLD,USD_MRU
date,,,,,
2026-09-10,104.715,46.0719,1.1611,4391.7000,39.6801
2026-09-11,106.885,46.0719,1.1611,4384.1499,39.6801
2026-09-14,106.360,46.2481,1.1553,4338.7000,40.0300


## 1. Ingestion manifest

Every series carries its provenance from ingestion all the way to the model.

In [2]:
import json
from mru_forecast.config import resolve_path

manifest = json.loads((resolve_path("data_interim", config) / "ingestion_manifest.json").read_text())
pd.DataFrame([
    {"series": k, "provider": v["provider"], "symbol": v["symbol"], "rate_type": v["rate_type"],
     "role": v["role"], "derived": bool(v["derived_from"]), "rows": v["rows"],
     "first": v["first_date"], "last": v["last_date"]}
    for k, v in manifest["series"].items()
])

,series,provider,symbol,rate_type,role,derived,rows,first,last
0,USD_MRU,yahoo,MRU=X,indicative_mid_market,target,False,2048,2018-11-01,2026-09-14
1,EUR_USD,yahoo,EURUSD=X,indicative_mid_market,auxiliary,False,3046,2015-01-01,2026-09-14
2,BRENT_OIL,yahoo,BZ=F,futures_settlement,auxiliary,False,2942,2015-01-02,2026-09-14
3,GOLD,yahoo,GC=F,futures_settlement,auxiliary,False,2940,2015-01-02,2026-09-14
4,EUR_MRU,derived,,indicative_mid_market_derived,target,True,2048,2018-11-01,2026-09-14


## 2. Raw vs cleaned

The raw feed emits isolated corrupted values — an observed close of **2.00 MRU/USD** between two days at 36.2. The daily value is therefore the *median of the four OHLC fields*, which no single bad field can move, and a second pass removes the bad ticks that survive that.

In [3]:
from mru_forecast.data.ingestion import load_interim, to_wide

raw = to_wide(load_interim(config))
comparison = pd.DataFrame({
    "raw_min": raw.min(), "raw_max": raw.max(),
    "raw_max_daily_move_%": raw.pct_change().abs().max() * 100,
    "clean_min": frame.min(), "clean_max": frame.max(),
    "clean_max_daily_move_%": frame.pct_change().abs().max() * 100,
})
comparison.round(3)

,raw_min,raw_max,raw_max_daily_move_%,clean_min,clean_max,clean_max_daily_move_%
BRENT_OIL,19.965,126.115,22.843,25.335,123.480,22.843
EUR_MRU,21.486,47.114,81.361,36.104,47.114,4.008
EUR_USD,0.960,1.251,2.871,0.960,1.234,2.787
GOLD,1058.050,5367.050,6.542,1197.750,5367.050,6.542
USD_MRU,20.199,40.030,81.496,33.900,40.030,3.756


In [4]:
flags = pd.read_csv(resolve_path("data_processed", config) / "cleaning_flags.csv")
print(f"{len(flags)} observations removed as bad ticks")
flags.groupby(["series", "reason"]).size().to_frame("n")

49 observations removed as bad ticks


n
series    reason                      
BRENT_OIL isolated_reverting_spike   3
          rolling_median_outlier    24
EUR_MRU   isolated_reverting_spike   7
          rolling_median_outlier     4
USD_MRU   isolated_reverting_spike   7
          rolling_median_outlier     4

In [5]:
# The corrupted window that motivated iterative spike detection.
window = slice("2022-12-12", "2022-12-30")
pd.DataFrame({"raw": raw.loc[window, "USD_MRU"], "cleaned": frame.loc[window, "USD_MRU"]}).round(3)

,raw,cleaned
date,,
2022-12-12,37.670,37.67
2022-12-13,37.600,37.60
2022-12-14,37.600,37.60
2022-12-15,37.250,37.25
2022-12-16,31.270,37.25
2022-12-19,34.068,37.25
2022-12-20,31.310,37.25
2022-12-21,34.128,37.25
2022-12-22,37.210,37.21


## 3. Independent cross-check

A second, unrelated provider is sampled to verify the *level*. The gap is small but **systematically negative**, which is a real limit on what accuracy can mean here: two honest providers disagree by 1–3% on this pair.

In [6]:
from mru_forecast.data.ingestion import compare_with_cross_check

cross_path = resolve_path("data_interim", config) / "cross_check.csv"
if cross_path.exists():
    cross = pd.read_csv(cross_path, index_col="date", parse_dates=["date"])
    gap = compare_with_cross_check(frame, cross, tolerance_pct=2.0)
    display(gap.round(3))
    print(f"mean USD/MRU gap: {gap['USD_MRU_gap_pct'].mean():+.2f}%")
else:
    print("no cross-check file; run scripts/download_data.py")

,USD_MRU_gap_pct,EUR_USD_gap_pct,within_tolerance
date,,,
2024-03-05,-0.240,0.006,True
2024-05-01,-0.458,0.553,True
2024-06-24,-1.310,0.431,True
2024-08-16,-1.516,-0.037,True
2024-10-09,-1.038,-0.240,True
2024-12-10,-1.717,0.032,True
2025-02-13,-2.524,-0.041,False
2025-04-15,-1.599,-0.264,True
2025-06-10,-0.534,0.417,True


mean USD/MRU gap: -1.41%


## 4. Shape of the series

In [7]:
from mru_forecast.evaluation import eda

eda.describe_levels(frame[TARGETS])[[
    "count", "mean", "std", "min", "max", "flat_day_share",
    "total_change_pct", "annualised_drift_pct",
]].round(4)

,count,mean,std,min,max,flat_day_share,total_change_pct,annualised_drift_pct
USD_MRU,2053.0,37.4762,1.5646,33.9000,40.0300,0.2962,11.9720,1.4475
EUR_MRU,2053.0,41.8986,2.5279,36.1042,47.1136,0.1232,14.3278,1.7163


**The number that shapes everything downstream is `flat_day_share`.** USD/MRU does not move *at all* on roughly 30% of business days. This is a managed rate that steps, not a freely floating one — which is exactly why the random walk is such a hard baseline to beat here.

In [8]:
viz.line_chart(frame[TARGETS], title='USD/MRU and EUR/MRU', y_title='MRU per unit')

In [9]:
viz.line_chart(
    frame[TARGETS].div(frame[TARGETS].iloc[0]).mul(100),
    title="Indexed to 100 at the first observation", y_title="index", hover_format=".1f",
)

## 5. Calendar coverage

Gaps are business days with no quote, filled forward (never backward — that would leak the future).

In [10]:
eda.summarise_gaps(frame['USD_MRU'])

,gap_days,n
0,1,1641
1,3,411
